# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `production_2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [6]:
# Write your code below.
%load_ext dotenv
%dotenv 


In [7]:
import dask
dask.config.set({'dataframe.query-planning': True})
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [8]:
import os 
os.getenv('PRICE_DATA')

'../../05_src/data/prices/'

In [9]:
import os 
from glob import glob 


# Write your code below.
PRICE_DATA = os.getenv('PRICE_DATA')
print(PRICE_DATA)
parquet_files = glob(os.path.join(PRICE_DATA, "*/**/*.parquet"))
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

../../05_src/data/prices/


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Adjusted Close:
    
    - `returns`: (Adj Close / Adj Close_lag) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [29]:
# Function to add lags
def add_lags(df):
    df['Close_lag_1'] = df['Close'].shift(1)
    return df


In [32]:
# Write your code below.
import dask.dataframe as dd

def add_lags(df):
    df['Close_lag'] = df['Close'].shift(1)
    return df

# Function to compute returns based on 'Close'
def compute_returns(df):
    df['returns'] = (df['Close'] / df['Close_lag']) - 1
    return df

# Function to calculate high-low range
def compute_hi_lo_range(df):
    df['hi_lo_range'] = df['High'] - df['Low']
    return df

dd_feat = dd_px.groupby('ticker').apply(add_lags, meta=dd_px).\
                  groupby('ticker').apply(compute_returns, meta=dd_px).\
                  groupby('ticker').apply(compute_hi_lo_range, meta=dd_px)

print(dd_feat.head())


Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, sector, subsector, year]
Index: []


In [41]:
dd_rets

,Date,Open,High,Low,Close,Adj Close,Volume,sector,subsector,year,Close_lag_1,returns,positive_return
npartitions=11217,,,,,,,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,int64,string,string,int32,float64,float64,int64
,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...


In [46]:
dd_rets.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,sector,subsector,year,Close_lag_1,returns,positive_return
ticker,,,,,,,,,,,,,
IP,2006-01-03,31.789791,31.789791,30.921268,31.276150,15.694670,2985132,Materials,Paper & Plastic Packaging Products & Materials,2006,NaN,NaN,0
IP,2006-01-04,31.229454,31.546980,31.192099,31.518963,15.816513,2222948,Materials,Paper & Plastic Packaging Products & Materials,2006,31.276150,0.007764,1
IP,2006-01-05,31.350861,31.416233,31.192099,31.416233,15.764964,1702547,Materials,Paper & Plastic Packaging Products & Materials,2006,31.518963,-0.003259,0
IP,2006-01-06,31.565657,31.985909,31.378878,31.920538,16.018030,2205173,Materials,Paper & Plastic Packaging Products & Materials,2006,31.416233,0.016052,1
IP,2006-01-09,32.191364,32.191364,31.462929,31.668385,15.891500,2574593,Materials,Paper & Plastic Packaging Products & Materials,2006,31.920538,-0.007899,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
BXP,2001-12-24,38.000000,38.500000,38.000000,38.320000,13.684836,73700,Real Estate,Office REITs,2001,37.930000,0.010282,1
BXP,2001-12-26,37.750000,38.150002,37.700001,38.099998,13.815379,292000,Real Estate,Office REITs,2001,38.320000,-0.005741,0
BXP,2001-12-27,38.200001,38.500000,37.990002,37.990002,13.775488,126600,Real Estate,Office REITs,2001,38.099998,-0.002887,0


+ Convert the Dask data frame to a pandas data frame. 
+ Add a rolling average return calculation with a window of 10 days.
+ *Tip*: Consider using `.rolling(10).mean()`.

(3 pt)

In [47]:
df = dd_rets.compute()

In [48]:
type(df)

pandas.core.frame.DataFrame

In [49]:
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,sector,subsector,year,Close_lag_1,returns,positive_return
ticker,,,,,,,,,,,,,
IP,2024-01-02,36.250000,36.849998,36.209999,36.540001,35.669991,3461900,Materials,Paper & Plastic Packaging Products & Materials,2024,NaN,NaN,0
IP,2024-01-03,36.180000,36.549999,35.910000,36.349998,35.484512,2530400,Materials,Paper & Plastic Packaging Products & Materials,2024,36.540001,-0.005200,0
IP,2024-01-04,36.349998,36.689999,36.310001,36.470001,35.601654,2949400,Materials,Paper & Plastic Packaging Products & Materials,2024,36.349998,0.003301,1
IP,2024-01-05,36.480000,37.310001,36.400002,37.270000,36.382610,5272300,Materials,Paper & Plastic Packaging Products & Materials,2024,36.470001,0.021936,1
IP,2024-01-08,37.060001,37.709999,37.060001,37.660000,36.763321,2833700,Materials,Paper & Plastic Packaging Products & Materials,2024,37.270000,0.010464,1


In [57]:
df['rolling_avg_return'] = df['returns'].rolling(window=10).mean()
print(type(df))
print(df.head())

<class 'pandas.core.frame.DataFrame'>
             Date       Open       High        Low      Close  Adj Close  \
ticker                                                                     
IP     2024-01-02  36.250000  36.849998  36.209999  36.540001  35.669991   
IP     2024-01-03  36.180000  36.549999  35.910000  36.349998  35.484512   
IP     2024-01-04  36.349998  36.689999  36.310001  36.470001  35.601654   
IP     2024-01-05  36.480000  37.310001  36.400002  37.270000  36.382610   
IP     2024-01-08  37.060001  37.709999  37.060001  37.660000  36.763321   

         Volume     sector                                       subsector  \
ticker                                                                       
IP      3461900  Materials  Paper & Plastic Packaging Products & Materials   
IP      2530400  Materials  Paper & Plastic Packaging Products & Materials   
IP      2949400  Materials  Paper & Plastic Packaging Products & Materials   
IP      5272300  Materials  Paper & Pla

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No.
Calculating directly in Dask reduces pandas conversion overhead and maximizes the use of distributed computer resources.


## Criteria

|Criteria|Complete|Incomplete|
|---------------------|----|----|
|Calculations         |Calculations were done correctly.|Calculations were not done correctly.|
|Explanation of answer|Answer was concise and explained the learner's reasoning in depth.|Answer was not concise and did not explained the learner's reasoning in depth.|

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.